In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

In [ ]:
from pathlib import Path
import os

base_dir = Path.cwd()
search_roots = [base_dir]
if base_dir.parent != base_dir and base_dir.parent != Path("/"):
    search_roots.append(base_dir.parent)
search_roots += [
    Path(p)
    for p in [
        "/content",
        "/content/drive",
        "/content/drive/MyDrive",
        "/mnt",
        "/kaggle/working",
        "/drive/MyDrive",
    ]
    if Path(p).exists()
]

search_paths = []
seen = set()

def add_candidate(path):
    if path not in seen:
        seen.add(path)
        search_paths.append(path)


def safe_walk(root):
    try:
        for dirpath, dirnames, filenames in os.walk(root, topdown=True, followlinks=False, onerror=lambda e: None):
            yield Path(dirpath), dirnames, filenames
    except (OSError, PermissionError):
        return

for root in dict.fromkeys(search_roots):
    if not root.exists():
        continue

    add_candidate(root / "Visadataset.csv")
    add_candidate(root / "notebooks" / "Visadataset.csv")

    for dirpath, dirnames, filenames in safe_walk(root):
        if "Visadataset.csv" in filenames:
            add_candidate(dirpath / "Visadataset.csv")
        if "eda.ipynb" in filenames:
            notebook_dir = dirpath
            add_candidate(notebook_dir / "Visadataset.csv")
            add_candidate((notebook_dir / ".." / "notebooks" / "Visadataset.csv").resolve())

csv_path = next((path for path in search_paths if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Visadataset.csv not found. Checked paths: "
        + ", ".join(str(path) for path in search_paths)
        + ".\nCurrent working directory: "
        + str(base_dir)
        + ".\nAvailable root candidates: "
        + ", ".join(str(path) for path in search_roots)
        + ".\nIf you are running this notebook in a remote runtime, make sure the project files are mounted into the notebook environment."
    )

print(f"Loading dataset from: {csv_path}")
df = pd.read_csv(csv_path)


FileNotFoundError: Visadataset.csv not found. Checked paths: /content/Visadataset.csv, /content/notebooks/Visadataset.csv, /mnt/Visadataset.csv, /mnt/notebooks/Visadataset.csv. Ensure the notebook runtime has the repo mounted and the dataset file is available.